# Discrete Diffusion (LLaDA-style)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winstonsmith1897/DantinoX/blob/main/docs/notebooks/02_discrete_diffusion.ipynb)

This notebook trains a discrete masked-diffusion model (LLaDA-style) and generates text via iterative unmasking.

**Key concepts:**
- `DiscreteParadigm` — owns the masking, (1/t)-weighted loss, and reverse diffusion
- `noise_schedule` — `"linear"` | `"cosine"` | `"sqrt"`
- Bidirectional model (`causal=False`) — attends to both left and right context

In [ ]:
!pip install -q git+https://github.com/winstonsmith1897/DantinoX.git#egg=dantinox[all]

In [ ]:
import urllib.request, os
if not os.path.exists('tiny_shakespeare.txt'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        'tiny_shakespeare.txt'
    )

In [ ]:
import dantinox as dx
from dantinox.paradigms.diffusion.discrete import DiscreteParadigm, DiscreteConfig

model_cfg = dx.ModelConfig(
    dim=256, n_heads=4, head_size=64, num_blocks=4,
    vocab_size=200, causal=False   # bidirectional — REQUIRED for diffusion
)
diff_cfg = DiscreteConfig(
    noise_schedule='cosine',
    mask_token_id=4,
)

paradigm = DiscreteParadigm(model_cfg, diff_cfg)
print(paradigm)

In [ ]:
trainer = dx.Trainer(paradigm, dx.TrainingConfig(lr=3e-4, epochs=2, batch_size=16))
run_dir = trainer.fit('tiny_shakespeare.txt')
print('Done:', run_dir)

In [ ]:
import jax
import jax.numpy as jnp

model = dx.load(run_dir, paradigm=paradigm)
prompt = jnp.array([[1, 2, 3, 4, 5]])   # short prefix

tokens = paradigm.generate(
    model, prompt, jax.random.PRNGKey(42),
    seq_len=64,
    n_steps=50,
)
print('Generated token IDs:', tokens)

## Noise schedule comparison

Visualize how the three schedules differ in masking rate `p(t)` vs. corruption level `t`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

t = np.linspace(0, 1, 200)
schedules = {
    'linear': t,
    'cosine': 1 - np.cos(np.pi * t / 2),
    'sqrt':   np.sqrt(t),
}

fig, ax = plt.subplots(figsize=(7, 4))
for name, p in schedules.items():
    ax.plot(t, p, label=name, linewidth=2)
ax.set_xlabel('t (corruption level)')
ax.set_ylabel('p_mask(t)')
ax.set_title('LLaDA Noise Schedules')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()